# Estimacion de grid

Este notebook carga los valores OTU positivos y propone una grilla logaritmica para las evaluaciones KDE posteriores.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import gaussian_kde
from sklearn.model_selection import KFold
from sklearn.neighbors import KernelDensity

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "Kernel_Tests" else cwd
real_data_file = project_root / "Datos" / "otu_data_converted.csv"
synthetic_data_file = project_root / "Datos" / "Sinteticos" / "otu_data_synthetic_small.csv"


In [ ]:
def choose_data_file(use_synthetic=False):
    return synthetic_data_file if use_synthetic else real_data_file


def load_otu_positives(use_synthetic=False):
    data_file = choose_data_file(use_synthetic)
    if not data_file.exists():
        raise FileNotFoundError("No se encontro el archivo de datos configurado.")

    df = pd.read_csv(data_file)
    values = df.select_dtypes(include=[np.number]).to_numpy(dtype=float).ravel()
    values = values[np.isfinite(values)]
    positives = values[values > 0]
    label = "sintetico" if use_synthetic else "real"
    return positives, df.shape, label


def suggest_grid_size(n_values):
    if n_values <= 250_000:
        return 1000
    if n_values <= 1_000_000:
        return 1500
    return 2000


def make_log_grid(values, grid_size, bandwidth):
    positives = np.asarray(values, dtype=float)
    positives = positives[np.isfinite(positives)]
    positives = positives[positives > 0]
    if positives.size == 0:
        raise ValueError("Se requiere al menos un valor positivo.")
    if bandwidth <= 0:
        raise ValueError("El bandwidth debe ser positivo.")

    lower = max(float(np.min(positives)) * 1e-3, 1e-12)
    upper = float(np.max(positives) + 8.0 * bandwidth)
    return np.logspace(np.log10(lower), np.log10(upper), int(grid_size))


In [ ]:
# Configuracion
USE_SYNTHETIC_DATA = False
BANDWIDTH_FOR_GRID = 100.0

values, data_shape, data_label = load_otu_positives(USE_SYNTHETIC_DATA)
grid_suggestion = suggest_grid_size(len(values))

summary = pd.DataFrame([{
    "datos": data_label,
    "filas": data_shape[0],
    "columnas": data_shape[1],
    "valores_positivos": len(values),
    "grid_sugerido": grid_suggestion,
}])
display(summary)


In [ ]:
grid_size = grid_suggestion
x_grid = make_log_grid(values, grid_size, BANDWIDTH_FOR_GRID)

grid_summary = pd.DataFrame([{
    "grid_size": grid_size,
    "minimo_grid": float(x_grid.min()),
    "maximo_grid": float(x_grid.max()),
    "minimo_datos": float(values.min()),
    "maximo_datos": float(values.max()),
}])
display(grid_summary)

print("Valor sugerido para copiar:")
print(f"grid_size = {grid_size}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(values, bins=80, density=True, alpha=0.35, label="Valores positivos")
ax.scatter(x_grid, np.zeros_like(x_grid), s=4, alpha=0.25, label="Grilla")
ax.set_xscale("log")
ax.set_xlabel("Valor OTU positivo")
ax.set_ylabel("Densidad empirica")
ax.set_title("Grilla logaritmica propuesta")
ax.grid(True, alpha=0.25)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()
